In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler, OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.feature_selection import SelectFromModel

# Preprocessing Pipeline Plan

- Limit observations to only the first encounter
- Drop additional prescribed medication due to limited clinical relevance
- Conduct feature decomposition on insulin and metformin to include a binary flag for being prescribed the medication and an ordinal encoding for the change
- Create train, validation and test splits
- Preprocess features
    - Convert data types (admission source, admission type)
    - Scale numeric features
    - Encode categoricals (OHE and ordinal)
    - Encode target (multi-class, binary)

## Experimenting with Feature Decomposition

Decomposing the insulin and metformin columns into two each: one binary flag to indicate if someone is on the drug at all, another ordinal encoding encoding if the patient is on the drug to indicate if there was a change in dosing.

In [ ]:
features = pd.read_parquet("data/diabetes_features.parquet")
target = pd.read_parquet("data/diabetes_target.parquet")

### Limiting the Data to the First Encounter Only

In [ ]:
# Creating a dataframe for selecting first encounters
splitting_df = features.copy()
splitting_df['target'] = (target['readmitted'] == "<30").astype(int)

# Keeping only the first encounter per patient_nbr
split_df = splitting_df.drop_duplicates('patient_nbr', keep='first')

### Preparing Features

Binary and Ordinal Splits for Insulin and Metformin

In [ ]:
# Creating a feature to indicate if a patient is not on insulin or metformin
split_df['missing_insulin'] = (split_df.insulin == "No").astype(int)
split_df['missing_metformin'] = (split_df.metformin == "No").astype(int)

# Creating a dictionary to encode changes in insulin and metformin
encode_dict = {'No': 0,
               "Steady": 0,
               "Up": 1,
               "Down": -1}

# Applying the encoding to engineer new features
split_df['metformin_change'] = split_df['metformin'].apply(lambda x: encode_dict[x])
split_df['insulin_change'] = split_df['insulin'].apply(lambda x: encode_dict[x])

Binary missingness encoding for weight

In [ ]:
split_df['missing_weight'] = (split_df['weight'] == "?").astype("int")

Binary indicator for missingness to one-hot encode:
- medical_specialty
- payer_code

In [ ]:
# Utility function for encoding missing values in a column for one-hot encoding
def missing_cleaner(x, missing_code, encoding=""):
    if x == missing_code:
        return encoding
    else:
        return x

split_df['payer_code_cleaned'] = split_df['payer_code'].apply(missing_cleaner, args=("?", "missing_payer"))
split_df['medical_specialty_cleaned'] = split_df['medical_specialty'].apply(missing_cleaner, args=("?", "missing_medical_specialty"))

Binary indicator and ordinal encoding for A1C and max_glu_serum

In [ ]:
split_df['missing_a1c'] = (split_df['A1Cresult'] == "None").astype("int")
split_df['missing_max_glu_serum'] = (split_df['max_glu_serum'] == "None").astype("int")

#### Creating Train and Test Splits

In [ ]:
# List of features to drop
drop_features = ['patient_nbr', 'encounter_id', 'target',
                 'repaglinide', 'nateglinide', 'chlorpropamide',
                 'glimepiride', 'acetohexamide', 'glipizide',
                 'glyburide', 'tolbutamide',
                 'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone',
                 'tolazamide', 'examide', 'citoglipton', 'insulin',
                 'glyburide.metformin', 'glipizide.metformin',
                 'glimepiride.pioglitazone', 'metformin.rosiglitazone',
                 'metformin.pioglitazone', 'metformin', 'insulin',
                 'A1Cresult', 'weight', 'diag_1', 'diag_2', 'diag_3',
                 'max_glu_serum', 'change']

# Building X & y dataframes
y = split_df['target']
X = split_df.drop(columns=drop_features)

# Creating train and test splits
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42, test_size=0.2)

## Setting up the Pipeline

Features for scaling:
- time_in_hospital
- num_lab_procedures
- num_procedures
- num_medications
- number_outpatient
- number_inpatient
- number_diagnoses

Features for OHE:
- race
- gender
- admission_type_id
- dicharge_disposition_id
- admission_source_id
- payer_code
- medical_specialty

Features for ordinal encoding:
- Age
- Metformin (already encoded)
- Insulin (already encoded)

#### OHE Columns

In [ ]:
OHEFEATURES = ['race', 'gender', 'admission_type_id',
               'discharge_disposition_id', 'admission_source_id',
               'payer_code', 'medical_specialty']

# Pipeline for OHE
ohe_pipeline = Pipeline([('handle_outliers', SimpleImputer(strategy='most_frequent')),
                         ('encoding', OneHotEncoder())])

#### Columns for Scaling

In [ ]:
SCALING_FEATURES = ['time_in_hospital', 'num_lab_procedures', 'num_procedures',
                    'num_medications', 'number_outpatient', 'number_inpatient', 'number_diagnoses']

# Pipeline for Scaling
scaling_pipeline = Pipeline([('scaling', MinMaxScaler())])

#### Columns for Ordinal Encoding

In [ ]:
# Setting up a list of columns for ordinal encoding
ORDINAL_COLS = ['age']

# Creating a list of categories for the age column
age_categories = sorted(split_df['age'].unique())

# Setting up a pipeline for ordinal encoding
ordinal_pipeline = Pipeline([('ordinal_encoding', OrdinalEncoder(categories=[age_categories]))])

#### Setting up the Column Transformer

In [ ]:
ct = ColumnTransformer([('OHE', ohe_pipeline, OHEFEATURES),
                        ('scaling', scaling_pipeline, SCALING_FEATURES),
                        ('ordinal_encoding', ordinal_pipeline, ORDINAL_COLS)])

## Fitting the Model

Next Step: Solve for missing columns in the test set that are present in the train set due to many different categories (i.e. medical specialty, payer_code, etc.)

In [ ]:
# Setting up a dictionary of models
models = {'random_forest': RandomForestClassifier(),
          'hist_gb': HistGradientBoostingClassifier(),
          'log_reg' : LogisticRegression()}

# Creating the parameter grid
param_grid = {
    'random_forest': {
        'model__n_estimators': [100, 200],
        'model__max_depth': [None, 5, 10],
        'model__min_samples_split': [2, 5]
    },
    'hist_gb' : {
        'model__max_depth': [None, 5, 10],
        'model__learning_rate': np.linspace(start=0.01, stop=1, num=5),
        'model__max_iter': np.arange(start=10, stop=100, step=25)
    },
    'log_reg': {
        'model__C': np.logspace(-3, 2, 6)
    }}

In [ ]:
cv = StratifiedKFold(n_splits=5,
                     random_state=42,
                     shuffle=True)

In [ ]:
cv_results = {}

# Setting up the loop to evaluate the different models across the parameter grids
for model_name, model in models.items():

    # Pipeline for model training
    model_training_pipeline = Pipeline([('preprocessing', ct),
                                        ('model', model)])

    # Setting up the grid for CV
    grid = GridSearchCV(model_training_pipeline,
                        param_grid=param_grid[model_name],
                        scoring=['average_precision', 'roc_auc', 'recall', 'precision'],
                        refit='average_precision',
                        cv=cv)

    # Storing the results
    cv_results[model_name] = grid.fit(X_train, y_train)

In [ ]:
cv_results